# 24. Swap Nodes in Pairs

[Problem](https://leetcode.com/problems/swap-nodes-in-pairs/) · difficulty: medium

The statement forbids a technique rather than a result: *"without modifying the values in the
list's nodes"*. That is invisible to any comparison of output values, so most of this notebook is
about how to test a rule the output cannot show.


In [ ]:
import pathlib, sys

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / 'lc').is_dir())
PROBLEM = ROOT / 'problems' / '0024-swap-nodes-in-pairs'
sys.path.insert(0, str(ROOT))

from lc.harness import load_module, load_solutions

solutions = load_solutions(PROBLEM)
tests = load_module(next(PROBLEM.glob('test_*.py')))
linked_list, to_list, ListNode = tests.linked_list, tests.NORMALIZE, tests.ListNode
solve = solutions[0]().swapPairs
[s.__name__ for s in solutions]


## Summary

| Approach | Time | Space | LeetCode |
|---|---|---|---|
| `SolutionRecursiveSwap` | O(n) | O(n) — one frame per pair | 0 ms, 19.14 MB |

Depth is `n/2`: 50 frames at the constraint's 100 nodes.


## The rewiring

With `first` and `second` at the front: `second` becomes the head, `first` follows it, and the
recursion's answer for the rest is attached behind `first`. Traced one pair at a time.


In [ ]:
def trace(values):
    def swap(head, depth=0):
        pad = '  ' * depth
        if head is None or head.next is None:
            print(f'{pad}base case: {to_list(head)}')
            return head
        first, second = head, head.next
        print(f'{pad}swap {first.val} and {second.val}, recurse on {to_list(second.next)}')
        first.next = swap(second.next, depth + 1)
        second.next = first
        print(f'{pad}-> {to_list(second)}')
        return second

    return to_list(swap(linked_list(values)))

trace([1, 2, 3, 4, 5])


## Testing a rule the output cannot show

A solution that swaps `val` fields produces byte-identical output to one that relinks nodes.
Both of these "work" if you only look at the answer.


In [ ]:
class SwapsValues:
    """Forbidden: the nodes never move, only the numbers in them."""

    def swapPairs(self, head):
        node = head
        while node is not None and node.next is not None:
            node.val, node.next.val = node.next.val, node.val
            node = node.next.next
        return head


values = [1, 2, 3, 4]
print('relinking :', to_list(solve(linked_list(values))))
print('value swap:', to_list(SwapsValues().swapPairs(linked_list(values))))
print('identical output, and only one of them is a legal answer')


The difference is visible in *which object* comes back. A correct solution returns the node that
was second; a value-swapper returns the original head.


In [ ]:
def identity_report(fn, values):
    head = linked_list(values)
    original_head = head
    original_second = head.next
    result = fn(head)
    return {
        'returns the old head': result is original_head,
        'returns the old second': result is original_second,
        'old head is now second': result.next is original_head,
    }

for label, fn in (('relinking', solve), ('value swap', SwapsValues().swapPairs)):
    print(f'{label:<12} {identity_report(fn, [1, 2, 3, 4])}')


That is exactly what the last two cases in the test module assert. Running the whole suite
against the cheating implementation shows the split: it passes everything that looks at values,
and fails only the identity checks.


In [ ]:
from lc.harness import Comparison, as_cases, check

cases = as_cases(tests.CASES)
comparison = Comparison(normalize=tests.NORMALIZE)

for cls in (solutions[0], SwapsValues):
    passed = failed = 0
    for case in cases:
        try:
            check(cls, case, comparison)
            passed += 1
        except AssertionError:
            failed += 1
    print(f'{cls.__name__:<22} passed {passed:>2}   failed {failed}')


## What the `holder` node costs

The solution allocates a `ListNode` per pair purely to hold a reference to `second` while the
links are rewritten. A local variable does the same job — same algorithm, one fewer object per
pair. The iterative form goes further and keeps a single sentinel for the whole list, which also
drops the space bound from O(n) to O(1).


In [ ]:
import timeit

def recursive_local(head):
    """The same recursion, holding `second` in a local instead of a node."""
    if head is None or head.next is None:
        return head
    second = head.next
    head.next = recursive_local(second.next)
    second.next = head
    return second


def iterative(head):
    """One sentinel for the whole list: O(1) space, no recursion."""
    sentinel = ListNode(0, head)
    prev = sentinel
    while prev.next is not None and prev.next.next is not None:
        first, second = prev.next, prev.next.next
        first.next = second.next
        second.next = first
        prev.next = second
        prev = first
    return sentinel.next


variants = {'as written (holder per pair)': solve,
            'local variable instead': recursive_local,
            'iterative (one sentinel)': iterative}

def swap_micros(fn, size):
    values = list(range(size))
    total = min(timeit.repeat(lambda: fn(linked_list(values)), number=100, repeat=5))
    build = min(timeit.repeat(lambda: linked_list(values), number=100, repeat=5))
    return (total - build) / 100 * 1e6

print(f"{'variant':<32}{'100 nodes':>14}{'1000 nodes':>14}")
for name, fn in variants.items():
    print(f'{name:<32}{swap_micros(fn, 100):11.1f} us{swap_micros(fn, 1000):11.1f} us')


## Takeaway

- When a statement forbids a *technique* rather than a result, the test cannot look at the result.
  Nine of twelve cases here pass a solution that cheats; only object identity catches it.
- A recursive solution is never O(1) space. One frame per pair is O(n), and the stack is as real
  as the heap — depth `n/2` is fine at 100 nodes and overflows around 2000.
- Allocating an object to hold a reference costs 2.8x here. Python names are already references.
